### [문제] 타이타닉 데이터(./titanic/train.csv)를 사용하여 아래 처리 조건을 먼저 수행 한 다음 VIF를 사용하여 다중공선성을 제거한 피처를 학습한 모델을 작성하여 해당 모델의 요약정보(summary())를 출력하는 프로그램을 완성하시오.
### (단, 최적의 결과가 나오도록 최선을 다해보시오!!!)

### <처리조건>
- target : Survived
- Passengerid, name, Ticket, Cabin 컬름은 삭제
- 누락된 데이터(NaN)가 있는 행은 삭제
- Sex, Emabrked는 레이블 인코딩

### 파일 불러오기

In [1]:
import pandas as pd

df = pd.read_csv("train.csv", encoding='utf-8')
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


### Passengerid, name, Ticket, Cabin 컬름은 삭제

In [2]:
df = df.drop(columns = ['PassengerId', 'Name', 'Ticket', 'Cabin'])
df.head(3)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    str    
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB


### 누락된 데이터(NaN)가 있는 행은 삭제

In [4]:
df = df.dropna(axis=0)
df.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Sex       712 non-null    str    
 3   Age       712 non-null    float64
 4   SibSp     712 non-null    int64  
 5   Parch     712 non-null    int64  
 6   Fare      712 non-null    float64
 7   Embarked  712 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 50.1 KB


### Sex, Emabrked는 레이블 인코딩

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])
df.head(3)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2


In [6]:
df.corr()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
Survived,1.000000,-0.356462,-0.536762,-0.082446,-0.015523,0.095265,0.266100,-0.181979
Pclass,-0.356462,1.000000,0.150826,-0.365902,0.065187,0.023666,-0.552893,0.244145
Sex,-0.536762,0.150826,1.000000,0.099037,-0.106296,-0.249543,-0.182457,0.109639
Age,-0.082446,-0.365902,0.099037,1.000000,-0.307351,-0.187896,0.093143,-0.032565
SibSp,-0.015523,0.065187,-0.106296,-0.307351,1.000000,0.383338,0.139860,0.033064
Parch,0.095265,0.023666,-0.249543,-0.187896,0.383338,1.000000,0.206624,0.011803
Fare,0.266100,-0.552893,-0.182457,0.093143,0.139860,0.206624,1.000000,-0.283510
Embarked,-0.181979,0.244145,0.109639,-0.032565,0.033064,0.011803,-0.283510,1.000000


### 일단 돌려봄 결정계수 결과 0.400

In [7]:
import statsmodels.formula.api as smf

formula = "Survived~" + "+".join(df.iloc[:,1:].columns)

model_titanic = smf.ols(formula=formula, data = df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               Survived   R-squared:                       0.400
Model:                            OLS   Adj. R-squared:                  0.394
Method:                 Least Squares   F-statistic:                     67.16
Date:                Tue, 28 Apr 2026   Prob (F-statistic):           4.63e-74
Time:                        19:52:48   Log-Likelihood:                -321.43
No. Observations:                 712   AIC:                             658.9
Df Residuals:                     704   BIC:                             695.4
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.4026      0.081     17.212      0.000       1.243       1.563
Pclass        -0.1903      0.023     -8.403      0.000      -0.235      -0.146
Sex           -0.4842      0.032    -15.368      0.000      -0.546      -0.422
Age           -0.0065      0.001     -5.724      0.000      -0.009      -0.004
SibSp         -0.0519      0.017     -2.982      0.003      -0.086      -0.018
Parch         -0.0102      0.019     -0.535      0.593      -0.048       0.027
Fare           0.0002      0.000      0.589      0.556      -0.000       0.001
Embarked      -0.0298      0.019     -1.535      0.125      -0.068       0.008
==============================================================================
Omnibus:                       25.995   Durbin-Watson:                   1.848
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               28.157
Skew:                           0.487   Prob(JB):                     7.69e-07
Kurtosis:                       3.027   Cond. No.                         386.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### vif 구해주는 함수 만들기

In [8]:
from patsy import dmatrices 
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def get_vif(formula, df):
    y, X = dmatrices(formula, df, return_type="dataframe")
    vif = pd.DataFrame()
    vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    vif["features"] = X.columns
    return vif

### formula 넣어줘서 vif 확인

In [9]:
formula = "Survived~" + "+".join(df.iloc[:,1:].columns)
print(formula)
get_vif(formula,df)

Survived~Pclass+Sex+Age+SibSp+Parch+Fare+Embarked


,VIF Factor,features
0,32.368412,Intercept
1,1.749120,Pclass
2,1.119651,Sex
3,1.310796,Age
4,1.276469,SibSp
5,1.288619,Parch
6,1.641348,Fare
7,1.114880,Embarked


### Intercept 제거

In [10]:
formula = "Survived~" + "+".join(df.iloc[:,1:].columns) + "-1"
print(formula)
get_vif(formula,df)

Survived~Pclass+Sex+Age+SibSp+Parch+Fare+Embarked-1


,VIF Factor,features
0,5.990097,Pclass
1,3.035750,Sex
2,4.169982,Age
3,1.638657,SibSp
4,1.618635,Parch
5,1.659422,Fare
6,5.282948,Embarked


### 결정계수 0.493

In [11]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data = df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.493
Model:                            OLS   Adj. R-squared (uncentered):              0.488
Method:                 Least Squares   F-statistic:                              97.81
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                    1.58e-99
Time:                        19:52:48   Log-Likelihood:                         -446.48
No. Observations:                 712   AIC:                                      907.0
Df Residuals:                     705   BIC:                                      938.9
Df Model:                           7                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Pclass         0.1069      0.017      6.121      0.000       0.073       0.141
Sex           -0.4207      0.037    -11.288      0.000      -0.494      -0.347
Age            0.0056      0.001      5.342      0.000       0.004       0.008
SibSp         -0.0132      0.021     -0.644      0.520      -0.054       0.027
Parch         -0.0023      0.023     -0.100      0.920      -0.047       0.042
Fare           0.0034      0.000      9.856      0.000       0.003       0.004
Embarked       0.0698      0.022      3.161      0.002       0.026       0.113
==============================================================================
Omnibus:                       53.614   Durbin-Watson:                   1.855
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               35.947
Skew:                           0.431   Prob(JB):                     1.56e-08
Kurtosis:                       2.316   Cond. No.                         149.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Sex:Age 추가 
결정계수 : 0.54 

vif 수치 증가

In [21]:
formula = "Survived~Pclass+Sex*Age+SibSp+Parch+Fare+Embarked-1"
print(formula)
get_vif(formula,df)

Survived~Pclass+Sex*Age+SibSp+Parch+Fare+Embarked-1


,VIF Factor,features
0,7.472861,Pclass
1,12.141691,Sex
2,9.388199,Age
3,13.514910,Sex:Age
4,1.648567,SibSp
5,1.618654,Parch
6,1.839843,Fare
7,5.414010,Embarked


In [22]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data = df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.540
Model:                            OLS   Adj. R-squared (uncentered):              0.535
Method:                 Least Squares   F-statistic:                              103.4
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                   1.78e-113
Time:                        19:54:49   Log-Likelihood:                         -411.36
No. Observations:                 712   AIC:                                      838.7
Df Residuals:                     704   BIC:                                      875.3
Df Model:                           8                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Pclass         0.0362      0.019      1.948      0.052      -0.000       0.073
Sex            0.1046      0.071      1.473      0.141      -0.035       0.244
Age            0.0153      0.002     10.107      0.000       0.012       0.018
Sex:Age       -0.0188      0.002     -8.543      0.000      -0.023      -0.014
SibSp         -0.0262      0.020     -1.336      0.182      -0.065       0.012
Parch         -0.0016      0.022     -0.076      0.939      -0.044       0.041
Fare           0.0025      0.000      7.151      0.000       0.002       0.003
Embarked       0.0415      0.021      1.949      0.052      -0.000       0.083
==============================================================================
Omnibus:                       39.395   Durbin-Watson:                   1.846
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               35.776
Skew:                           0.486   Prob(JB):                     1.70e-08
Kurtosis:                       2.488   Cond. No.                         298.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Sex:Age, Sex 
결정계수 : 0.474

vif 수치 감소

In [24]:
formula = "Survived~Pclass+Sex:Age+Sex+SibSp+Parch+Fare+Embarked-1"
print(formula)
get_vif(formula,df)

Survived~Pclass+Sex:Age+Sex+SibSp+Parch+Fare+Embarked-1


,VIF Factor,features
0,5.782513,Pclass
1,6.002954,Sex:Age
2,8.271139,Sex
3,1.636270,SibSp
4,1.617218,Parch
5,1.362981,Fare
6,4.841085,Embarked


In [25]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data = df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.474
Model:                            OLS   Adj. R-squared (uncentered):              0.468
Method:                 Least Squares   F-statistic:                              90.62
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                    6.28e-94
Time:                        19:56:18   Log-Likelihood:                         -459.60
No. Observations:                 712   AIC:                                      933.2
Df Residuals:                     705   BIC:                                      965.2
Df Model:                           7                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Pclass         0.1255      0.017      7.181      0.000       0.091       0.160
Sex:Age       -0.0022      0.002     -1.414      0.158      -0.005       0.001
Sex           -0.3005      0.063     -4.797      0.000      -0.424      -0.178
SibSp         -0.0434      0.021     -2.074      0.038      -0.084      -0.002
Parch         -0.0081      0.023     -0.353      0.724      -0.053       0.037
Fare           0.0043      0.000     13.361      0.000       0.004       0.005
Embarked       0.1116      0.022      5.179      0.000       0.069       0.154
==============================================================================
Omnibus:                       61.675   Durbin-Watson:                   1.837
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               31.739
Skew:                           0.349   Prob(JB):                     1.28e-07
Kurtosis:                       2.237   Cond. No.                         233.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Sex:Age, Age 
결정계수 : 0.539

vif 수치 감소

In [28]:
formula = "Survived~Pclass+Sex:Age+Age+SibSp+Parch+Fare+Embarked-1"
print(formula)
get_vif(formula,df)

Survived~Pclass+Sex:Age+Age+SibSp+Parch+Fare+Embarked-1


,VIF Factor,features
0,5.302315,Pclass
1,3.379092,Sex:Age
2,6.395410,Age
3,1.637864,SibSp
4,1.599769,Parch
5,1.687061,Fare
6,5.225288,Embarked


In [29]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data = df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.539
Model:                            OLS   Adj. R-squared (uncentered):              0.534
Method:                 Least Squares   F-statistic:                              117.7
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                   4.62e-114
Time:                        19:57:02   Log-Likelihood:                         -412.45
No. Observations:                 712   AIC:                                      838.9
Df Residuals:                     705   BIC:                                      870.9
Df Model:                           7                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Pclass         0.0509      0.016      3.252      0.001       0.020       0.082
Sex:Age       -0.0160      0.001    -14.522      0.000      -0.018      -0.014
Age            0.0140      0.001     11.229      0.000       0.012       0.016
SibSp         -0.0239      0.020     -1.221      0.223      -0.062       0.015
Parch         -0.0051      0.021     -0.236      0.813      -0.047       0.037
Fare           0.0026      0.000      7.905      0.000       0.002       0.003
Embarked       0.0474      0.021      2.262      0.024       0.006       0.088
==============================================================================
Omnibus:                       36.098   Durbin-Watson:                   1.845
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               33.970
Skew:                           0.480   Prob(JB):                     4.20e-08
Kurtosis:                       2.528   Cond. No.                         98.4
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""